# Experiment 2 — Onboarding check: are mules linked to each other?

**Question (BCB indicator ii):** could a new account be flagged *on the day it opens*, before any credit, because its
customer is linked to a known mule through `CRMUSER.ENTITYRELATIONSHIP`?

Two things decide whether that check is worth building:

1. **Would it have caught mules?** For each of the 13 mules, was its CIF linked to another mule's CIF — directly, or through
   a shared party (the same director, shareholder, guarantor…)? Links to a mule that opened *earlier* are the ones an
   onboarding check could have acted on.
2. **What would it cost?** How many accounts sit within reach of the mules' CIFs, and how many such accounts open each month —
   that is the alert volume of an onboarding check.

**How links are found** (your relationship query, run for all mules at once):

| Hop | From | Through `ENTITYRELATIONSHIP` | To |
|---|---|---|---|
| 1 | a mule's CIF | either side of a relationship row | a **party** CIF (director, shareholder, guarantor…) |
| 2 | a party CIF | either side of a relationship row | another **customer** sharing that party |

Parties linked to very many customers (`HUB_MAX`) are reported but not followed — a law firm or group holding company would
otherwise link everything to everything.

**Scope and safety:** read-only, separate from production. Small queries (only the mules' CIFs and their neighbours), cached.

In [15]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd

SYNTHETIC = False

NEW_MULES = ["UG6302468", "UG6300154", "UG6334629"]  # same identifiers as the scan notebook (ACID or account number)
INCLUDE_WATCHLIST = True                              # add the active watchlist mules

HUB_MAX = 25            # a party linked to more customers than this is reported, not followed to hop 2
ER_DATE_COL = "BODATECREATED"     # set to a date column of ENTITYRELATIONSHIP (printed below) to check links existed at opening
SINCE = "2024-01-01"    # alert-volume window for the onboarding check

AS_OF = pd.Timestamp.today().normalize()
OUT = Path("outputs/network_onboarding"); OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("cache/network_onboarding" + ("_synthetic" if SYNTHETIC else "")); CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 50)

## 1. Connections and queries

In [16]:
ENV_FILE = ".env"
try:
    from dotenv import load_dotenv
    load_dotenv(ENV_FILE, override=False)
except ImportError:
    print("python-dotenv not installed; relying on variables already in the environment")

def _engine(prefix):
    '''Copied from aml_screening.py. READ = Finacle, WRITE = reporting DB (SELECTs only here).'''
    from sqlalchemy import create_engine, URL
    full_url = os.environ.get(f"AML_{prefix}_URL")
    if full_url:
        return create_engine(full_url, fast_executemany=True)
    def req(name):
        val = os.environ.get(f"AML_{prefix}_{name}")
        if not val:
            raise RuntimeError(f"Set AML_{prefix}_{name} (or provide a full AML_{prefix}_URL)")
        return val
    url = URL.create("mssql+pyodbc", username=req("USER"), password=req("PWD"), host=req("HOST"),
                     port=int(os.environ.get(f"AML_{prefix}_PORT", "1433")), database=req("DB"),
                     query={"driver": os.environ.get("AML_ODBC_DRIVER", "ODBC Driver 17 for SQL Server")})
    return create_engine(url, fast_executemany=True)

_fin = _rep = None
def fin():
    global _fin, _rep
    if _fin is None:
        _fin, _rep = _engine("READ"), _engine("WRITE")
    return _fin
def rep():
    fin(); return _rep

GAM = "RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE"
ER = "RSVRRealtimeFinacleUGPRD.CRMUSER.ENTITYRELATIONSHIP"
ACCT_COLS = "ACID, CIF_ID, FORACID, ACCT_NAME, ACCT_CRNCY_CODE, ACCT_OPN_DATE, ACCT_CLS_FLG"

MULE_SQL = f"SELECT {ACCT_COLS} FROM {GAM} WHERE ACID IN ({{ids}}) OR FORACID IN ({{ids}})"
ACCTS_BY_CIF_SQL = f"SELECT {ACCT_COLS} FROM {GAM} WHERE CIF_ID IN ({{ids}})"
ER_COLS_SQL = f"SELECT TOP 0 * FROM {ER}"
EDGES_SQL = f'''
SELECT PARENTCIFID, CHILDCIFID, RELATIONSHIP{{date_col}}
FROM {ER}
WHERE (PARENTCIFID IN ({{ids}}) OR CHILDCIFID IN ({{ids}}))
  AND PARENTCIFID IS NOT NULL AND CHILDCIFID IS NOT NULL AND PARENTCIFID <> CHILDCIFID
'''

def _in_list(ids):
    ids = [str(i).strip() for i in ids]
    bad = [i for i in ids if not re.fullmatch(r"[A-Za-z0-9_\-]+", i)]
    if bad:
        raise ValueError(f"Unexpected identifier format: {bad[:5]}")
    return ",".join(f"'{i}'" for i in ids)

def read_chunks(template, ids, eng, chunk=500, **kw):
    ids = list(dict.fromkeys(str(i) for i in ids))
    parts = [pd.read_sql(template.format(ids=_in_list(ids[i:i + chunk]), **kw), eng) for i in range(0, len(ids), chunk)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

## 2. Synthetic data (dry-run only)

In [17]:
def make_synthetic(seed=5):
    '''Fake customers, accounts and relationship rows. Planted: two mules sharing a director, a mule that is a shareholder
    of another mule, and one hub party linked to many companies. Not evidence of anything.'''
    rng = np.random.default_rng(seed)
    cifs = [f"C{i:07d}" for i in range(3000)]
    accts = []
    for i, c in enumerate(cifs):
        for j in range(rng.integers(1, 3)):
            accts.append(dict(ACID=f"UG{len(accts):07d}", CIF_ID=c, FORACID=f"9{len(accts):09d}",
                              ACCT_NAME=f"SYNTHETIC CUSTOMER {i:05d}", ACCT_CRNCY_CODE=rng.choice(["USD", "UGX"], p=[.4, .6]),
                              ACCT_OPN_DATE=pd.Timestamp("2022-01-01") + pd.Timedelta(days=int(rng.integers(0, 1750))),
                              ACCT_CLS_FLG="N"))
    gam = pd.DataFrame(accts)
    first = gam.groupby("CIF_ID").ACID.first()
    mules = [first[cifs[k]] for k in range(13)]
    for k, a in enumerate(mules):                                   # mules are USD, opened since 2024, in order
        gam.loc[gam.ACID == a, ["ACCT_CRNCY_CODE", "ACCT_OPN_DATE", "ACCT_NAME"]] = \
            ["USD", pd.Timestamp("2024-02-01") + pd.Timedelta(days=60 * k), f"SYNTHETIC MULE {k:02d}"]
    rels = ["Beneficial Owner", "ShareHolder", "CorpRep", "Guarantor"]
    rows = []
    def edge(p, c, r, d):
        rows.append(dict(PARENTCIFID=p, CHILDCIFID=c, RELATIONSHIP=r, RCRE_TIME=d))
    persons = cifs[2000:]
    for c in cifs[:2000]:                                           # companies get 1-2 parties
        for p in rng.choice(persons, rng.integers(1, 3), replace=False):
            edge(c, p, rng.choice(rels), pd.Timestamp("2022-01-01") + pd.Timedelta(days=int(rng.integers(0, 1700))))
    edge(cifs[1], "C0002999", "CorpRep", pd.Timestamp("2024-01-15"))     # mules 01 and 05 share a director
    edge(cifs[5], "C0002999", "Beneficial Owner", pd.Timestamp("2024-09-20"))
    edge(cifs[8], cifs[3], "ShareHolder", pd.Timestamp("2025-03-01"))    # mule 03 holds shares in mule 08
    for c in cifs[100:400]:                                         # a hub party (e.g. a group holding company)
        edge(c, "C0002500", "ShareHolder", pd.Timestamp("2023-01-01"))
    edge(cifs[10], "C0002500", "ShareHolder", pd.Timestamp("2023-01-01"))
    return gam, pd.DataFrame(rows), mules[3:], mules[:3]

## 3. Load the mules, their relationship rows, and their neighbours

In [18]:
def cached(name, fn):
    p = CACHE_DIR / f"{name}.pkl"
    if p.exists():
        return pd.read_pickle(p)
    df = fn(); df.to_pickle(p); return df

if SYNTHETIC:
    syn_gam, syn_er, syn_watch, syn_new = make_synthetic()
    NEW_MULES = syn_new
    def q_accts_by_ids(ids):  return syn_gam[syn_gam.ACID.isin(ids) | syn_gam.FORACID.isin(ids)].copy()
    def q_accts_by_cif(cifs): return syn_gam[syn_gam.CIF_ID.isin(cifs)].copy()
    def q_edges(cifs):
        e = syn_er[syn_er.PARENTCIFID.isin(cifs) | syn_er.CHILDCIFID.isin(cifs)]
        return e.rename(columns={ER_DATE_COL: "REL_DATE"}) if ER_DATE_COL else e.drop(columns=["RCRE_TIME"])
    watch = list(syn_watch)
    er_cols = list(syn_er.columns)
else:
    def q_accts_by_ids(ids):  return read_chunks(MULE_SQL, ids, fin())
    def q_accts_by_cif(cifs): return read_chunks(ACCTS_BY_CIF_SQL, cifs, fin())
    def q_edges(cifs):
        dc = f", {ER_DATE_COL} AS REL_DATE" if ER_DATE_COL else ""
        return read_chunks(EDGES_SQL, cifs, fin(), date_col=dc)
    watch = pd.read_sql("SELECT acid FROM aml_watchlist WHERE active = 1", rep())["acid"].tolist() if INCLUDE_WATCHLIST else []
    er_cols = list(pd.read_sql(ER_COLS_SQL, fin()).columns)

print("ENTITYRELATIONSHIP columns that look like dates (candidates for ER_DATE_COL):",
      [c for c in er_cols if re.search(r"DATE|TIME|_ON$|_DT$", c.upper())] or "none")

ids = list(dict.fromkeys([str(x) for x in NEW_MULES] + [str(x) for x in watch]))
tag = f"{len(ids)}_{ER_DATE_COL}"
mules = cached(f"mules_{tag}", lambda: q_accts_by_ids(ids))
mules["ACCT_OPN_DATE"] = pd.to_datetime(mules.ACCT_OPN_DATE)
newset = {str(x) for x in NEW_MULES}
mules["mule_group"] = np.where(mules.ACID.astype(str).isin(newset) | mules.FORACID.astype(str).isin(newset), "new", "reference")
mule_cifs = set(mules.CIF_ID.astype(str))
print(f"Mules found: {len(mules)} accounts, {len(mule_cifs)} distinct CIFs")
same = mules[mules.duplicated("CIF_ID", keep=False)]
if len(same):
    print("Mules that are the same customer (same CIF):"); display(same[["ACCT_NAME", "CIF_ID", "ACCT_OPN_DATE"]])

def other_end(edges, cifs):
    '''Rows touching `cifs`, oriented as (from_cif in cifs) -> (to_cif).'''
    rows = []
    for col_a, col_b in (("PARENTCIFID", "CHILDCIFID"), ("CHILDCIFID", "PARENTCIFID")):
        e = edges[edges[col_a].astype(str).isin(cifs)]
        rows.append(pd.DataFrame({"from_cif": e[col_a].astype(str), "to_cif": e[col_b].astype(str),
                                  "relationship": e.RELATIONSHIP,
                                  "rel_date": pd.to_datetime(e["REL_DATE"]) if "REL_DATE" in e else pd.NaT}))
    return pd.concat(rows, ignore_index=True).drop_duplicates()

hop1 = other_end(cached(f"edges1_{tag}", lambda: q_edges(sorted(mule_cifs))), mule_cifs)
parties = set(hop1.to_cif)
print(f"Hop 1: {len(hop1)} relationship rows from the mules to {len(parties)} party CIFs")

edges2 = cached(f"edges2_{tag}", lambda: q_edges(sorted(parties)))
hop2_all = other_end(edges2, parties)
fan = hop2_all.groupby("from_cif").to_cif.nunique().rename("customers_linked")
hubs = set(fan[fan > HUB_MAX].index)
hop2 = hop2_all[~hop2_all.from_cif.isin(hubs)]
print(f"Hop 2: {len(hop2_all)} rows from the parties; {len(hubs)} hub part{'y' if len(hubs) == 1 else 'ies'} "
      f"(> {HUB_MAX} customers) not followed")

ENTITYRELATIONSHIP columns that look like dates (candidates for ER_DATE_COL): ['BODATECREATED', 'BODATEMODIFIED', 'TMDATE']
Mules found: 13 accounts, 13 distinct CIFs
Hop 1: 29 relationship rows from the mules to 23 party CIFs
Hop 2: 56 rows from the parties; 0 hub parties (> 25 customers) not followed


## 4. Would an onboarding check have caught mules?

A link counts when two mules are joined **directly** (one mule's CIF is a party of the other) or through a **shared party**.
`earlier_opened` marks links to a mule that opened before this one: those are the cases where, had the earlier mule already
been on the watchlist, an onboarding check could have flagged this one on its opening day.

In [23]:
cif_info = (mules.sort_values("ACCT_OPN_DATE").groupby("CIF_ID")
                 .agg(name=("ACCT_NAME", "first"), opened=("ACCT_OPN_DATE", "min"), group=("mule_group", "first")))
cif_info.index = cif_info.index.astype(str)

links = []
for _, r in hop1[hop1.to_cif.isin(mule_cifs)].iterrows():                     # direct mule-to-mule
    links.append(dict(mule_cif=r.from_cif, linked_cif=r.to_cif, path=f"direct: {r.relationship}", via=None,
                      rel_date=r.rel_date))
for _, a in hop1[~hop1.to_cif.isin(mule_cifs)].iterrows():                    # shared party
    if a.to_cif in hubs:
        continue
    for _, b in hop2[(hop2.from_cif == a.to_cif) & hop2.to_cif.isin(mule_cifs) & (hop2.to_cif != a.from_cif)].iterrows():
        links.append(dict(mule_cif=a.from_cif, linked_cif=b.to_cif, via=a.to_cif,
                          path=f"shared party: {a.relationship} / {b.relationship}",
                          rel_date=max([d for d in (a.rel_date, b.rel_date) if pd.notna(d)], default=pd.NaT)))
L = pd.DataFrame(links, columns=["mule_cif", "linked_cif", "path", "via", "rel_date"])
if len(L):
    L["mule"] = L.mule_cif.map(cif_info.name); L["mule_opened"] = L.mule_cif.map(cif_info.opened)
    L["linked_mule"] = L.linked_cif.map(cif_info.name); L["linked_opened"] = L.linked_cif.map(cif_info.opened)
    L["earlier_opened"] = L.linked_opened < L.mule_opened
    if ER_DATE_COL:
        L["link_existed_at_opening"] = L.rel_date <= L.mule_opened
    L = L.drop_duplicates(["mule_cif", "linked_cif", "path", "via"])
    display(L.drop(columns=["mule_cif", "linked_cif"]).sort_values("mule_opened"))
else:
    print("No mule is linked to another mule, directly or through a shared party.")

caught = set(L.loc[L.earlier_opened, "mule_cif"]) if len(L) else set()
if ER_DATE_COL and len(L):
    caught = set(L.loc[L.earlier_opened & L.link_existed_at_opening, "mule_cif"])
linked_any = set(L.mule_cif) | set(L.linked_cif) if len(L) else set()

,path,via,rel_date,mule,mule_opened,linked_mule,linked_opened,earlier_opened,link_existed_at_opening
0,shared party: Beneficial Owner / Beneficial Owner,10302424758,2025-02-20 12:05:30,PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,2024-09-20,KWELIVEST INVESTMENT SMC LIMITED,2025-02-20,False,False
1,shared party: Beneficial Owner / Beneficial Owner,10302424758,2025-02-20 13:49:21,PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,2024-09-20,AETHERGRID TECHNOLOGIES LIMITED,2025-02-20,False,False
2,shared party: Beneficial Owner / Beneficial Owner,10302424758,2025-02-20 13:50:26,PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,2024-09-20,NEBULACORE INNOVATIONS LIMITED,2025-02-20,False,False
3,shared party: Beneficial Owner / Contact,10302111252,2025-02-07 14:22:08,HESHIMA PRIME LOGISTICS LTD,2025-01-06,EMIC VENTURES,2025-02-07,False,False
4,shared party: Contact / Beneficial Owner,10302111252,2025-02-07 14:22:08,EMIC VENTURES,2025-02-07,HESHIMA PRIME LOGISTICS LTD,2025-01-06,True,False
17,shared party: Beneficial Owner / Beneficial Owner,10302424758,2025-02-20 13:50:26,NEBULACORE INNOVATIONS LIMITED,2025-02-20,PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,2024-09-20,True,False
16,shared party: Beneficial Owner / Beneficial Owner,10302496871,2025-02-20 13:50:26,NEBULACORE INNOVATIONS LIMITED,2025-02-20,AETHERGRID TECHNOLOGIES LIMITED,2025-02-20,False,False
15,shared party: Beneficial Owner / Beneficial Owner,10302496871,2025-02-20 13:50:26,NEBULACORE INNOVATIONS LIMITED,2025-02-20,KWELIVEST INVESTMENT SMC LIMITED,2025-02-20,False,False
14,shared party: Beneficial Owner / Beneficial Owner,10302496871,2025-02-20 13:50:26,AETHERGRID TECHNOLOGIES LIMITED,2025-02-20,NEBULACORE INNOVATIONS LIMITED,2025-02-20,False,False
13,shared party: Beneficial Owner / Beneficial Owner,10302496871,2025-02-20 13:49:21,AETHERGRID TECHNOLOGIES LIMITED,2025-02-20,KWELIVEST INVESTMENT SMC LIMITED,2025-02-20,False,False


In [24]:
print(f"\nMules (CIFs) linked to any other mule: {len(linked_any)} of {len(mule_cifs)}")
print(f"Mules an onboarding check could have flagged on opening day (linked to a mule that opened earlier"
      f"{' and link recorded by then' if ER_DATE_COL else ''}): {len(caught)} of {len(mule_cifs)}")
if not ER_DATE_COL:
    print("Note: without ER_DATE_COL we cannot tell whether each link was already recorded when the account opened.")


Mules (CIFs) linked to any other mule: 6 of 13
Mules an onboarding check could have flagged on opening day (linked to a mule that opened earlier and link recorded by then): 0 of 13


## 5. What would it cost? Accounts within reach of the mules

An onboarding check flags any **new** account whose customer is a party of a watchlist mule (hop 1) or shares a party with
one (hop 2). Below: how many customers and accounts sit within reach today, and how many of those accounts opened per month
since `SINCE` — the alert volume such a check would have produced.

In [20]:
reach1 = parties - mule_cifs
reach2 = set(hop2.to_cif) - mule_cifs - parties
reach_accts = cached(f"reach_{tag}_{HUB_MAX}", lambda: q_accts_by_cif(sorted(reach1 | reach2)))
reach_accts["ACCT_OPN_DATE"] = pd.to_datetime(reach_accts.ACCT_OPN_DATE)
reach_accts["hop"] = np.where(reach_accts.CIF_ID.astype(str).isin(reach1), "1: a mule's party", "2: shares a party")

recent = reach_accts[reach_accts.ACCT_OPN_DATE >= SINCE]
months = max((AS_OF.to_period("M") - pd.Timestamp(SINCE).to_period("M")).n, 1)
summary = pd.DataFrame({
    "customers in reach": [len(reach1), len(reach2), len(reach1 | reach2)],
    "accounts in reach (any age)": [int((reach_accts.hop.str[0] == "1").sum()), int((reach_accts.hop.str[0] == "2").sum()), len(reach_accts)],
    f"opened since {SINCE}": [int((recent.hop.str[0] == "1").sum()), int((recent.hop.str[0] == "2").sum()), len(recent)],
    "of which USD": [int(((recent.hop.str[0] == "1") & (recent.ACCT_CRNCY_CODE == "USD")).sum()),
                     int(((recent.hop.str[0] == "2") & (recent.ACCT_CRNCY_CODE == "USD")).sum()),
                     int((recent.ACCT_CRNCY_CODE == "USD").sum())],
}, index=["hop 1: the mules' parties", "hop 2: shares a party", "total"])
summary["alerts per month (all currencies)"] = (summary[f"opened since {SINCE}"] / months).round(1)
display(summary)
print(f"Over {months} months. Open (not closed) accounts in reach today: {int((reach_accts.ACCT_CLS_FLG != 'Y').sum()):,}")

,customers in reach,accounts in reach (any age),opened since 2024-01-01,of which USD,alerts per month (all currencies)
hop 1: the mules' parties,23,23,15,8,0.5
hop 2: shares a party,21,48,33,15,1.0
total,44,71,48,23,1.5


Over 33 months. Open (not closed) accounts in reach today: 46


## 6. Who the parties are

In [21]:
print("Relationship types from the mules to their parties:")
print(hop1.relationship.value_counts().to_string())
per_mule = hop1.groupby("from_cif").agg(parties=("to_cif", "nunique"), types=("relationship", lambda s: ", ".join(sorted(set(map(str, s))))))
per_mule.index = per_mule.index.map(lambda c: cif_info.name.get(c, c))
print("\nParties per mule:"); display(per_mule.reindex(cif_info.name.values).fillna({"parties": 0}))
if hubs:
    print("\nHub parties (not followed to hop 2):")
    display(fan[fan.index.isin(hubs)].sort_values(ascending=False).to_frame())

Relationship types from the mules to their parties:
relationship
Beneficial Owner    24
Contact              4
ShareHolder          1

Parties per mule:


,parties,types
from_cif,,
PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,2,Beneficial Owner
HESHIMA PRIME LOGISTICS LTD,2,Beneficial Owner
EMIC VENTURES,1,Contact
KWELIVEST INVESTMENT SMC LIMITED,2,Beneficial Owner
AETHERGRID TECHNOLOGIES LIMITED,2,Beneficial Owner
NEBULACORE INNOVATIONS LIMITED,2,Beneficial Owner
SHAUPAM ENTERPRISES LIMITED,3,Beneficial Owner
FRONTIER TEXTILE DEALERS CO LIMITED,3,"Beneficial Owner, ShareHolder"
THRIFTOPIA UGANDA LIMITED,2,Beneficial Owner


## 7. Export

In [22]:
L.to_csv(OUT / "mule_links.csv", index=False)
summary.to_csv(OUT / "onboarding_reach_summary.csv")
reach_accts.to_csv(OUT / "accounts_in_reach.csv", index=False)
hop1.to_csv(OUT / "mule_parties.csv", index=False)
print("Saved to", OUT.resolve())

Saved to /home/A270827/experiments/AML/outputs/network_onboarding


## 8. Reading the result

| What you see | Reading |
|---|---|
| Several mules linked to an earlier mule, and modest alerts per month | Strong case for an onboarding check: it acts before any credit, as BCB asked |
| Mules linked, but alerts per month large | The check is real but needs narrowing (strong relationship types only, hop 1 only, USD only) |
| No mule linked to another | Linkage doesn't recur among these mules; the onboarding check adds little for now, and the early-life rule remains the main proactive control |

Two cautions: a link to a mule that opened earlier only helps if that mule had **already been confirmed** by then (we use
opening order as a stand-in), and without a date on the relationship rows we can't prove the link was recorded at onboarding.